In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
new_df = pd.read_csv("train.csv")

In [3]:
new_df.dropna(inplace=True)

In [4]:
new_df.drop_duplicates(inplace=True)

In [5]:
import re
import string

CONTRACTIONS = {
    "can't": "cannot",
    "won't": "will not",
    "n't": " not",
    "'re": " are",
    "'s": " is",
    "'d": " would",
    "'ll": " will",
    "'t": " not",
    "'ve": " have",
    "'m": " am",
}


def preprocess_text(text, stem=False):
    """Clean raw text and prepare it for feature extraction.

    Steps:
    - lowercasing
    - normalize contractions
    - remove HTML tags, URLs, emails, mentions, and hashtags
    - remove punctuation, digits, and extra whitespace
    - optional stemming when nltk is available
    """
    if pd.isna(text):
        return ""

    text = str(text).lower().strip()

    for old, new in CONTRACTIONS.items():
        text = text.replace(old, new)

    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"\S+@\S+", " ", text)
    text = re.sub(r"[@#]\w+", " ", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\d+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    tokens = text.split()

    if stem:
        try:
            from nltk.stem import PorterStemmer
            stemmer = PorterStemmer()
            tokens = [stemmer.stem(token) for token in tokens]
        except Exception:
            pass

    return " ".join(tokens)


sample_text = "<p>How can I improve my coding skills? Visit https://example.com now! I can't wait :)</p>"
preprocess_text(sample_text)

'how can i improve my coding skills visit now i cannot wait'

In [6]:
new_df['question1'] = new_df['question1'].apply(preprocess_text)
new_df['question2'] = new_df['question2'].apply(preprocess_text)

In [7]:
new_df.head()

,id,qid1,qid2,question1,question2,is_duplicate
0,0,1,2,what is the step by step guide to invest in sh...,what is the step by step guide to invest in sh...,0
1,1,3,4,what is the story of kohinoor kohinoor diamond,what would happen if the indian government sto...,0
2,2,5,6,how can i increase the speed of my internet co...,how can internet speed be increased by hacking...,0
3,3,7,8,why am i mentally very lonely how can i solve it,find the remainder when math math is divided by,0
4,4,9,10,which one dissolve in water quikly sugar salt ...,which fish would survive in salt water,0


In [8]:
new_df.drop(columns=['id', 'qid1', 'qid2'], axis=1)

,question1,question2,is_duplicate
0,what is the step by step guide to invest in sh...,what is the step by step guide to invest in sh...,0
1,what is the story of kohinoor kohinoor diamond,what would happen if the indian government sto...,0
2,how can i increase the speed of my internet co...,how can internet speed be increased by hacking...,0
3,why am i mentally very lonely how can i solve it,find the remainder when math math is divided by,0
4,which one dissolve in water quikly sugar salt ...,which fish would survive in salt water,0
...,...,...,...
404285,how many keywords are there in the racket prog...,how many keywords are there in perl programmin...,0
404286,do you believe there is life after death,is it true that there is life after death,1
404287,what is one coin,what is this coin,0
404288,what is the approx annual cost of living while...,i am having little hairfall problem but i want...,0


Using Transformer

In [10]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

emb1 = model.encode(new_df['question1'].tolist(), batch_size=64)
emb2 = model.encode(new_df['question2'].tolist(), batch_size=64)

c:\Users\Acer\anaconda3\envs\py310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Acer\anaconda3\envs\py310\lib\site-packages\google\api_core\_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


Absolute difference

In [11]:
diff = np.abs(emb1 - emb2)

Element-wise multiplication

In [12]:
mul = emb1 * emb2

Cosine similarity (single score)

In [14]:
from sklearn.metrics.pairwise import cosine_similarity
# Compute row-wise cosine similarity (no NxN allocation)
num = np.sum(emb1 * emb2, axis=1, dtype=np.float32)
den = np.linalg.norm(emb1, axis=1) * np.linalg.norm(emb2, axis=1)
cos = (num / np.clip(den, 1e-12, None)).reshape(-1, 1).astype(np.float32)

Final feature matrix

In [15]:
X = np.hstack([diff, mul, cos])

In [16]:
y = new_df['is_duplicate'].values

In [17]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [19]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier()
rf.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [20]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = rf.predict(X_test)

print(accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

0.797311335922234
              precision    recall  f1-score   support

           0       0.82      0.86      0.84     51026
           1       0.74      0.69      0.71     29832

    accuracy                           0.80     80858
   macro avg       0.78      0.77      0.78     80858
weighted avg       0.80      0.80      0.80     80858



In [21]:
import pickle

pickle.dump(rf, open('Hybrid_model.pkl', 'wb'))